# JobScope — NLP & Feature Extraction


## 1. Data Source Inspection
Inspect the LinkedIn Kaggle dataset and verify available source files.

In [66]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data/raw/linkedin_kaggle")

print("Current directory:", Path.cwd())
print("Data directory exists:", DATA_DIR.exists())

csv_files = list(DATA_DIR.rglob("*.csv"))

for file in csv_files:
    print(file, f"({file.stat().st_size / 1e6:.1f} MB)")

Current directory: /Users/jessy/Documents/nyu/career/github_project/Py3onistas/notebooks
Data directory exists: True
../data/raw/linkedin_kaggle/postings.csv (516.8 MB)
../data/raw/linkedin_kaggle/mappings/skills.csv (0.0 MB)
../data/raw/linkedin_kaggle/mappings/industries.csv (0.0 MB)
../data/raw/linkedin_kaggle/jobs/benefits.csv (1.9 MB)
../data/raw/linkedin_kaggle/jobs/salaries.csv (2.3 MB)
../data/raw/linkedin_kaggle/jobs/job_industries.csv (2.5 MB)
../data/raw/linkedin_kaggle/jobs/job_skills.csv (3.5 MB)
../data/raw/linkedin_kaggle/companies/company_industries.csv (0.8 MB)
../data/raw/linkedin_kaggle/companies/company_specialities.csv (4.4 MB)
../data/raw/linkedin_kaggle/companies/companies.csv (23.2 MB)
../data/raw/linkedin_kaggle/companies/employee_counts.csv (1.0 MB)


## 2. Job Posting Schema
Identify job IDs, titles, descriptions, and relevant metadata.

In [67]:

import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/raw/linkedin_kaggle")

# 10 lines 
postings = pd.read_csv(
    DATA_DIR / "postings.csv",
    nrows=10
)

print("Columns:")
print(postings.columns.tolist())

print("\nFirst 3 jobs:")
display(postings.head(3))

Columns:
['job_id', 'company_name', 'title', 'description', 'max_salary', 'pay_period', 'location', 'company_id', 'views', 'med_salary', 'min_salary', 'formatted_work_type', 'applies', 'original_listed_time', 'remote_allowed', 'job_posting_url', 'application_url', 'application_type', 'expiry', 'closed_time', 'formatted_experience_level', 'skills_desc', 'listed_time', 'posting_domain', 'sponsored', 'work_type', 'currency', 'compensation_type', 'normalized_salary', 'zip_code', 'fips']

First 3 jobs:


,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,...,Requirements: \n\nWe are seeking a College or ...,1.713398e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,38480.0,8540.0,34021.0
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,...,NaN,1.712858e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,83200.0,80521.0,8069.0
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,...,We are currently accepting resumes for FOH - A...,1.713278e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,55000.0,45202.0,39061.0


In [68]:
print("JD sample:")
print(postings["description"].iloc[0][:1500])

print("\nJob IDs:")
print(postings["job_id"].tolist())

JD sample:
Job descriptionA leading real estate firm in New Jersey is seeking an administrative Marketing Coordinator with some experience in graphic design. You will be working closely with our fun, kind, ambitious members of the sales team and our dynamic executive team on a daily basis. This is an opportunity to be part of a fast-growing, highly respected real estate brokerage with a reputation for exceptional marketing and extraordinary culture of cooperation and inclusion.Who you are:You must be a well-organized, creative, proactive, positive, and most importantly, kind-hearted person. Please, be responsible, respectful, and cool-under-pressure. Please, be proficient in Adobe Creative Cloud (Indesign, Illustrator, Photoshop) and Microsoft Office Suite. Above all, have fantastic taste and be a good-hearted, fun-loving person who loves working with people and is eager to learn.Role:Our office is a fast-paced environment. You’ll work directly with a Marketing team and communicate dai


## 3. Existing Skill Mapping Analysis
Examine LinkedIn skill-category mappings and job-skill relationships.

In [69]:
# Inspect existing LinkedIn skill mappings
skills = pd.read_csv(DATA_DIR / "mappings/skills.csv")

job_skills = pd.read_csv(
    DATA_DIR / "jobs/job_skills.csv",
    nrows=10
)

print("SKILL MAPPINGS")
display(skills.head(10))

print("\nJOB-SKILL RELATIONSHIPS")
display(job_skills.head(10))

SKILL MAPPINGS


,skill_abr,skill_name
0,ART,Art/Creative
1,DSGN,Design
2,ADVR,Advertising
3,PRDM,Product Management
4,DIST,Distribution
5,EDU,Education
6,TRNG,Training
7,PRJM,Project Management
8,CNSL,Consulting
9,PRCH,Purchasing



JOB-SKILL RELATIONSHIPS


,job_id,skill_abr
0,3884428798,MRKT
1,3884428798,PR
2,3884428798,WRT
3,3887473071,SALE
4,3887465684,FIN
5,3887465684,SALE
6,3887467939,SALE
7,3887467939,ADVR
8,3887467939,BD
9,3887471331,ENG


## 4. Cross-Occupation Sampling
Select 10 job descriptions from different occupational domains for validation.

In [70]:
import pandas as pd

# Define target occupations for cross-domain validation
occupations = [
    "Data Scientist",
    "Data Analyst",
    "Accountant",
    "Financial Analyst",
    "Marketing Manager",
    "Sales Representative",
    "Project Manager",
    "Software Engineer",
    "Mechanical Engineer",
    "Registered Nurse"
]

samples = []
found_titles = set()

# Read the large dataset in manageable chunks
for chunk in pd.read_csv(
    DATA_DIR / "postings.csv",
    usecols=["job_id", "title", "description"],
    chunksize=10000
):
    chunk = chunk.dropna(subset=["description", "title"])

    for occupation in occupations:
        if occupation in found_titles:
            continue

        matches = chunk[
            chunk["title"].str.contains(
                occupation,
                case=False,
                regex=False,
                na=False
            )
        ]

        if not matches.empty:
            samples.append(matches.iloc[0])
            found_titles.add(occupation)

    if len(found_titles) == len(occupations):
        break

validation_df = pd.DataFrame(samples).reset_index(drop=True)

print(f"Collected {len(validation_df)} job descriptions")
display(validation_df[["job_id", "title"]])

Collected 10 job descriptions


,job_id,title
0,3884435309,Data Scientist Job
1,3813645405,Data Analyst
2,2428973832,Staff Accountant
3,3884432610,Financial Analyst
4,69333422,Senior Product Marketing Manager
5,1129235875,Industrial Sales Representative
6,2987175317,Construction Project Manager
7,133130219,Software Engineer
8,3619548798,Senior Mechanical Engineer
9,263583866,Registered Nurse


In [71]:
from pathlib import Path

# Create the reference directory if it does not exist
OUTPUT_DIR = Path("../data/reference")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_path = OUTPUT_DIR / "jd_validation_sample.csv"

# Save the selected job descriptions
validation_df.to_csv(output_path, index=False)

print(f"Saved validation dataset to: {output_path}")
print(f"Total samples: {len(validation_df)}")



Saved validation dataset to: ../data/reference/jd_validation_sample.csv
Total samples: 10


## 5. Job Description Preprocessing

This section implements and validates text preprocessing for
cross-occupation job descriptions.

In [72]:
import sys
from pathlib import Path

# Add the project root to the Python import path
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_extraction import clean_description

In [73]:
test_cases = {
    "HTML": "<p>Python &amp; SQL</p><br>Experience required.",
    "Empty text": "",
    "Missing value": None,
    "Extra whitespace": "Python   SQL\n\nExcel\tTableau",
    "Special skills": "C++, C#, .NET, R, Node.js",
    "Unicode": "Experience with Python—SQL and “Excel”.",
}

for name, text in test_cases.items():
    print(f"\nTest: {name}")
    print(f"Input: {text}")
    print(f"Output: {clean_description(text)}")


Test: HTML
Input: <p>Python &amp; SQL</p><br>Experience required.
Output: Python & SQL Experience required.

Test: Empty text
Input: 
Output: 

Test: Missing value
Input: None
Output: 

Test: Extra whitespace
Input: Python   SQL

Excel	Tableau
Output: Python SQL Excel Tableau

Test: Special skills
Input: C++, C#, .NET, R, Node.js
Output: C++, C#, .NET, R, Node.js

Test: Unicode
Input: Experience with Python—SQL and “Excel”.
Output: Experience with Python-SQL and "Excel".


In [74]:
import pandas as pd

validation_df = pd.read_csv(
    "../data/reference/jd_validation_sample.csv"
)

validation_df["cleaned_description"] = (
    validation_df["description"].apply(clean_description)
)

validation_df["original_length"] = validation_df["description"].str.len()
validation_df["cleaned_length"] = validation_df["cleaned_description"].str.len()

display(
    validation_df[
        ["job_id", "title", "original_length", "cleaned_length"]
    ]
)

,job_id,title,original_length,cleaned_length
0,3884435309,Data Scientist Job,6283,6246
1,3813645405,Data Analyst,1249,1247
2,2428973832,Staff Accountant,1475,1473
3,3884432610,Financial Analyst,4151,4153
4,69333422,Senior Product Marketing Manager,2151,2150
5,1129235875,Industrial Sales Representative,1723,1717
6,2987175317,Construction Project Manager,1005,984
7,133130219,Software Engineer,173,173
8,3619548798,Senior Mechanical Engineer,4205,4205
9,263583866,Registered Nurse,1596,1596


In [75]:
assert len(validation_df) == 10
assert validation_df["cleaned_description"].notna().all()
assert validation_df["cleaned_description"].str.len().gt(0).all()

assert clean_description("<p>C++ &amp; C#</p>") == "C++ & C#"
assert clean_description(None) == ""
assert clean_description("   ") == ""

print("All preprocessing checks passed!")

All preprocessing checks passed!


## 6. Cross-Occupation Skill Dictionary

This section loads and validates the curated skill dictionary
used for standardized skill extraction.

In [76]:
import json
from pathlib import Path
from collections import Counter

# Load the curated skill dictionary
dictionary_path = Path("../data/reference/skill_dictionary.json")

with open(dictionary_path, "r", encoding="utf-8") as file:
    skill_dictionary = json.load(file)

skills = skill_dictionary["skills"]

print("Dictionary version:", skill_dictionary["version"])
print("Number of skills:", len(skills))
print("Number of categories:", len(skill_dictionary["categories"]))
print("Number of aliases:", sum(len(s["aliases"]) for s in skills))

Dictionary version: 0.1.0
Number of skills: 95
Number of categories: 6
Number of aliases: 105


In [77]:
# Validate skill dictionary structure
required_fields = {"name", "category", "aliases"}

assert len(skills) == 95

assert all(
    required_fields.issubset(skill.keys())
    for skill in skills
)

# Check unique canonical skill names
skill_names = [skill["name"].casefold() for skill in skills]

assert len(skill_names) == len(set(skill_names))

# Check category distribution
category_counts = Counter(skill["category"] for skill in skills)

assert len(category_counts) == 6

print("Category distribution:")

for category, count in category_counts.items():
    print(f"{category}: {count}")

print("\nAll dictionary validation checks passed!")

Category distribution:
Data & Technology: 20
Finance & Accounting: 15
Marketing & Sales: 15
Business & Management: 15
Engineering: 15
Healthcare & Administration: 15

All dictionary validation checks passed!


In [78]:
# Display a sample of standardized skills and aliases
import pandas as pd

skill_df = pd.DataFrame(skills)

display(
    skill_df[
        ["name", "category", "aliases"]
    ].head(20)
)

,name,category,aliases
0,Python,Data & Technology,[Python 3]
1,SQL,Data & Technology,[Structured Query Language]
2,Excel,Data & Technology,"[Microsoft Excel, MS Excel]"
3,Tableau,Data & Technology,[Tableau Desktop]
4,Power BI,Data & Technology,[Microsoft Power BI]
5,R,Data & Technology,"[R language, R programming]"
6,Machine Learning,Data & Technology,[ML]
7,Deep Learning,Data & Technology,[DL]
8,Data Analysis,Data & Technology,[Data Analytics]
9,Data Visualization,Data & Technology,[Data Visualisation]


In [79]:
# Inspect selected cross-occupation skills
selected_skills = [
    "Python",
    "Excel",
    "C++",
    ".NET",
    "Financial Modeling",
    "SEO",
    "Project Management",
    "AutoCAD",
    "Electronic Medical Records"
]

display(
    skill_df[
        skill_df["name"].isin(selected_skills)
    ]
)

,name,category,aliases
0,Python,Data & Technology,[Python 3]
2,Excel,Data & Technology,"[Microsoft Excel, MS Excel]"
13,C++,Data & Technology,[CPP]
15,.NET,Data & Technology,"[dotnet, dot net]"
20,Financial Modeling,Finance & Accounting,[Financial Modelling]
35,SEO,Marketing & Sales,"[Search Engine Optimization, Search Engine Opt..."
50,Project Management,Business & Management,[Project Planning and Management]
66,AutoCAD,Engineering,[Auto CAD]
81,Electronic Medical Records,Healthcare & Administration,"[EMR, Electronic Health Records, EHR]"


## 7. Dictionary-Based Skill Extraction

Extract standardized skills using regex matching, alias normalization,
and deduplication.

Apply the extractor to 10 real job descriptions.

In [80]:
import sys
from pathlib import Path

# Identify the current Python environment
print("Python executable:", sys.executable)
print("Current directory:", Path.cwd())

# Locate the feature extraction module
PROJECT_ROOT = Path(
    "/Users/jessy/Documents/nyu/career/github_project/Py3onistas"
)

MODULE_PATH = PROJECT_ROOT / "src" / "feature_extraction.py"

print("Project exists:", PROJECT_ROOT.exists())
print("Module exists:", MODULE_PATH.exists())

Python executable: /usr/local/bin/python3
Current directory: /Users/jessy/Documents/nyu/career/github_project/Py3onistas/notebooks
Project exists: True
Module exists: True


In [81]:
import sys
import importlib
from pathlib import Path

# Add the project root to the Python import path
PROJECT_ROOT = Path(
    "/Users/jessy/Documents/nyu/career/github_project/Py3onistas"
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import the updated module
import src.feature_extraction as fe
importlib.reload(fe)

clean_description = fe.clean_description
extract_skills = fe.extract_skills

print("Module imported successfully!")

# Verify skill extraction
result = extract_skills(
    "Experience with Python, SQL, MS Excel, and Tableau."
)

print("Extracted skills:", result)

Module imported successfully!
Extracted skills: ['Python', 'SQL', 'Excel', 'Tableau']


In [82]:
import importlib
import src.feature_extraction as fe

# Reload the updated module
importlib.reload(fe)

extract_skills = fe.extract_skills

# Test dictionary-based skill extraction
sample_jd = """
We are looking for a Data Analyst proficient in
Python, SQL, MS Excel, and Tableau.
Experience with Python and Microsoft Excel is preferred.
"""

print("Extracted skills:")
print(extract_skills(sample_jd))

Extracted skills:
['Python', 'SQL', 'Excel', 'Tableau']


In [83]:
import pandas as pd

# Load the cross-occupation validation dataset
validation_df = pd.read_csv(
    "../data/reference/jd_validation_sample.csv"
)

# Extract standardized skills
validation_df["extracted_skills"] = (
    validation_df["description"].apply(extract_skills)
)

# Count extracted skills for each posting
validation_df["skill_count"] = (
    validation_df["extracted_skills"].apply(len)
)

display(
    validation_df[
        ["job_id", "title", "skill_count", "extracted_skills"]
    ]
)

,job_id,title,skill_count,extracted_skills
0,3884435309,Data Scientist Job,7,"[Machine Learning, Tableau, Statistics, Python..."
1,3813645405,Data Analyst,5,"[Statistics, Data Visualization, SQL, Python, R]"
2,2428973832,Staff Accountant,2,"[Reconciliation, Excel]"
3,3884432610,Financial Analyst,4,"[Strategic Planning, Risk Management, Excel, S..."
4,69333422,Senior Product Marketing Manager,1,[Market Research]
5,1129235875,Industrial Sales Representative,0,[]
6,2987175317,Construction Project Manager,0,[]
7,133130219,Software Engineer,0,[]
8,3619548798,Senior Mechanical Engineer,3,"[Manufacturing, SolidWorks, CAD]"
9,263583866,Registered Nurse,0,[]


In [84]:
# Summarize skill extraction results
total_jobs = len(validation_df)
matched_jobs = (validation_df["skill_count"] > 0).sum()
average_skills = validation_df["skill_count"].mean()

print(f"Total jobs: {total_jobs}")
print(f"Jobs with extracted skills: {matched_jobs}")
print(f"Average skills per job: {average_skills:.2f}")

# Inspect each job's extracted skills
for _, row in validation_df.iterrows():
    print(f"\nJob: {row['title']}")
    print(f"Skills: {row['extracted_skills']}")

Total jobs: 10
Jobs with extracted skills: 6
Average skills per job: 2.20

Job: Data Scientist Job
Skills: ['Machine Learning', 'Tableau', 'Statistics', 'Python', 'R', 'SQL', 'Manufacturing']

Job: Data Analyst
Skills: ['Statistics', 'Data Visualization', 'SQL', 'Python', 'R']

Job: Staff Accountant
Skills: ['Reconciliation', 'Excel']

Job: Financial Analyst
Skills: ['Strategic Planning', 'Risk Management', 'Excel', 'Statistics']

Job: Senior Product Marketing Manager
Skills: ['Market Research']

Job: Industrial Sales Representative
Skills: []

Job: Construction Project Manager
Skills: []

Job: Software Engineer
Skills: []

Job: Senior Mechanical Engineer
Skills: ['Manufacturing', 'SolidWorks', 'CAD']

Job: Registered Nurse
Skills: []


In [85]:
# Validate skill extraction behavior
assert extract_skills("") == []
assert extract_skills(None) == []

# Validate canonical names and aliases
result = extract_skills("Python, SQL, MS Excel, Tableau.")
assert set(result) == {"Python", "SQL", "Excel", "Tableau"}

# Validate deduplication
result = extract_skills(
    "Python, Python, MS Excel, Microsoft Excel."
)
assert result.count("Python") == 1
assert result.count("Excel") == 1

# Validate special skill symbols
result = extract_skills("C++, C#, and .NET.")
assert {"C++", "C#", ".NET"}.issubset(set(result))

print("All skill extraction checks passed!")

All skill extraction checks passed!


## 8. Manual Validation and Error Analysis

Evaluate the skill extraction model using 10 real job descriptions
across different occupational domains.

Objectives:
- Annotate expected skills using the curated skill dictionary.
- Identify true positives, false positives, and false negatives.
- Calculate micro-averaged Precision, Recall, and F1-score.
- Document dictionary coverage gaps and extraction limitations.

In [86]:
import pandas as pd
from pathlib import Path

# Prepare an independent annotation dataset
annotation_df = validation_df[
    ["job_id", "title", "description"]
].copy()

# Empty labels must not be treated as completed annotations
annotation_df["expected_skills"] = ""
annotation_df["reviewed"] = False
annotation_df["annotation_notes"] = ""

# Save the annotation template
ANNOTATION_PATH = Path(
    "../data/reference/jd_skill_annotations.csv"
)

if not ANNOTATION_PATH.exists():
    annotation_df.to_csv(ANNOTATION_PATH, index=False)
    print(f"Annotation template saved: {ANNOTATION_PATH}")
else:
    print("Existing annotation file preserved:", ANNOTATION_PATH)

# Display the current annotation file
current_annotations = pd.read_csv(
    ANNOTATION_PATH,
    encoding="utf-8"
)

display(
    current_annotations[
        ["job_id", "title", "expected_skills", "reviewed"]
    ]
)

Existing annotation file preserved: ../data/reference/jd_skill_annotations.csv


,job_id,title,expected_skills,reviewed
0,3884435309,Data Scientist Job,"[""Machine Learning"", ""Tableau"", ""Power BI"", ""D...",True
1,3813645405,Data Analyst,"[""Data Analysis"", ""Statistics"", ""Data Visualiz...",True
2,2428973832,Staff Accountant,"[""Accounts Payable"", ""Accounts Receivable"", ""R...",True
3,3884432610,Financial Analyst,"[""Financial Analysis"", ""Strategic Planning"", ""...",True
4,69333422,Senior Product Marketing Manager,"[""Market Research"", ""Budgeting""]",True
5,1129235875,Industrial Sales Representative,"[""Excel""]",True
6,2987175317,Construction Project Manager,"[""Budgeting"", ""Contract Management"", ""Vendor M...",True
7,133130219,Software Engineer,[],True
8,3619548798,Senior Mechanical Engineer,"[""SolidWorks"", ""CAD"", ""3D Modeling"", ""Manufact...",True
9,263583866,Registered Nurse,"[""Patient Care"", ""Medication Administration""]",True


In [87]:
import json
from pathlib import Path

# Load the skill dictionary independently
DICTIONARY_PATH = Path("../data/reference/skill_dictionary.json")

with open(DICTIONARY_PATH, "r", encoding="utf-8") as file:
    dictionary_data = json.load(file)

# Retrieve the list of standardized skills
skill_entries = dictionary_data["skills"]

# Extract canonical skill names
canonical_skills = sorted(
    entry["name"] for entry in skill_entries
)

print("Total canonical skills:", len(canonical_skills))
print("\nAvailable canonical skills:")
print(", ".join(canonical_skills))

# Display complete job descriptions for manual annotation
for _, row in validation_df.iterrows():
    print("\n" + "=" * 80)
    print(f"Job ID: {row['job_id']}")
    print(f"Job title: {row['title']}")
    print(f"Predicted skills: {row['extracted_skills']}")
    print("\nFull job description:")
    print(row["description"])

Total canonical skills: 95

Available canonical skills:
.NET, 3D Modeling, AWS, Accounts Payable, Accounts Receivable, Adobe Illustrator, Adobe InDesign, Adobe Photoshop, Agile, Auditing, AutoCAD, Blueprint Reading, Bookkeeping, Budgeting, Business Analysis, C#, C++, CAD, CPR, CRM, Change Management, Circuit Design, Clinical Documentation, Content Marketing, Contract Management, Copywriting, Data Analysis, Data Visualization, Deep Learning, Electrical Engineering, Electronic Medical Records, Email Marketing, Excel, Financial Analysis, Financial Modeling, Financial Reporting, Finite Element Analysis, First Aid, Forecasting, GAAP, General Ledger, Git, Google Analytics, HIPAA Compliance, HVAC, HubSpot, Infection Control, Inventory Management, Java, JavaScript, Lead Generation, Lean Manufacturing, MATLAB, Machine Learning, Manufacturing, Market Research, Mechanical Design, Medical Billing, Medical Coding, Medical Terminology, Medication Administration, Operations Management, PLC Programmin

In [88]:
# Review one job description at a time
job_index = 0

row = validation_df.iloc[job_index]

print("Job ID:", row["job_id"])
print("Job title:", row["title"])
print("Predicted skills:", row["extracted_skills"])
print("\nFull job description:")
print(row["description"])

Job ID: 3884435309
Job title: Data Scientist Job
Predicted skills: ['Machine Learning', 'Tableau', 'Statistics', 'Python', 'R', 'SQL', 'Manufacturing']

Full job description:
Primary location: Remote, Remote

Relocation offered: No

Employment status: Full-Time

Travel: <10%

Non-compete: No

Location: Work from home position located in Remote, Remote

The estimated base salary range for this role is $97,060 to $145,580 per year.

Individual pay is based upon location, skills and expertise, experience and other relevant factors (salary may be adjusted based on geographic location)

 What does it mean to work at Armstrong?

It means being immersed in a supportive culture that recognizes you as a key player in Armstrong's future. We are a large company with a local feel, where you will get to know and collaborate with leadership and your colleagues across the company.

By joining us, you'll have the opportunity to make the most of your potential. Alongside a competitive remuneration pack

In [89]:
import pandas as pd
from pathlib import Path

# Load the annotation file using UTF-8
annotation_path = Path("../data/reference/jd_skill_annotations.csv")

annotations = pd.read_csv(
    annotation_path,
    encoding="utf-8"
)

# Validate annotation completeness
assert len(annotations) == 10
assert annotations["reviewed"].all()
assert annotations["expected_skills"].notna().all()

print("Annotation file loaded successfully!")
print("Total jobs:", len(annotations))
print("Reviewed jobs:", annotations["reviewed"].sum())

Annotation file loaded successfully!
Total jobs: 10
Reviewed jobs: 10


In [90]:
import json
import pandas as pd

# Load manually reviewed annotations
annotations = pd.read_csv("../data/reference/jd_skill_annotations.csv")

assert len(annotations) == 10
assert annotations["reviewed"].all()

annotations["expected_skills"] = annotations["expected_skills"].apply(json.loads)

# Join ground truth with model predictions
evaluation_df = validation_df[
    ["job_id", "title", "extracted_skills"]
].merge(
    annotations[["job_id", "expected_skills"]],
    on="job_id",
    validate="one_to_one"
)

# Calculate true positives, false positives, and false negatives
evaluation_df["TP"] = evaluation_df.apply(
    lambda row: set(row["extracted_skills"]) & set(row["expected_skills"]),
    axis=1
)

evaluation_df["FP"] = evaluation_df.apply(
    lambda row: set(row["extracted_skills"]) - set(row["expected_skills"]),
    axis=1
)

evaluation_df["FN"] = evaluation_df.apply(
    lambda row: set(row["expected_skills"]) - set(row["extracted_skills"]),
    axis=1
)

tp = evaluation_df["TP"].apply(len).sum()
fp = evaluation_df["FP"].apply(len).sum()
fn = evaluation_df["FN"].apply(len).sum()

precision = tp / (tp + fp) if tp + fp else 0
recall = tp / (tp + fn) if tp + fn else 0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0

print(f"True Positives: {tp}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-score: {f1:.3f}")

display(evaluation_df[["title", "TP", "FP", "FN"]])

True Positives: 20
False Positives: 2
False Negatives: 17
Precision: 0.909
Recall: 0.541
F1-score: 0.678


,title,TP,FP,FN
0,Data Scientist Job,"{SQL, Python, R, Machine Learning, Statistics,...",{Manufacturing},"{Data Analysis, Power BI, MATLAB}"
1,Data Analyst,"{SQL, Python, R, Statistics, Data Visualization}",{},{Data Analysis}
2,Staff Accountant,"{Excel, Reconciliation}",{},"{Accounts Receivable, Accounts Payable, Auditing}"
3,Financial Analyst,"{Excel, Statistics, Strategic Planning}",{Risk Management},"{Budgeting, Financial Analysis}"
4,Senior Product Marketing Manager,{Market Research},{},{Budgeting}
5,Industrial Sales Representative,{},{},{Excel}
6,Construction Project Manager,{},{},"{Budgeting, Vendor Management, Contract Manage..."
7,Software Engineer,{},{},{}
8,Senior Mechanical Engineer,"{Manufacturing, CAD, SolidWorks}",{},{3D Modeling}
9,Registered Nurse,{},{},"{Patient Care, Medication Administration}"
